# VQ-VAE on 1D vectors -- pythae's built-in EMA quantizer

This version, unlike the first notebook, does **not** port the custom
`VectorQuantizerCodebook` / `VectorQuantizerLookup` logic from
`vqvae_quantizer.py`. Instead it uses **pythae's own `VQVAE` model with its
built-in `QuantizerEMA`** (`pythae.models.vq_vae.vq_vae_utils.QuantizerEMA`)
unmodified -- no subclassing of `VQVAE`, no custom quantizer module.

What *is* still ported from `vqvae_layers.py` is the **encoder/decoder
architecture** (`EncoderConv`/`DecoderConv`/`ResBlock`, as 1D convolutions),
since the data here is one-dimensional vectors (not images). The only change
needed to make that architecture "just work" with pythae's stock `VQVAE` is
that the encoder must return a plain 2D embedding `(batch, embedding_dim)` --
pythae's `VQVAE._set_quantizer` / `VQVAE.forward` special-case 2D encoder
output (reshaping it to `(N, 1, 1, D)` internally) and, from there on,
everything -- nearest-codebook lookup, EMA codebook updates, straight-through
gradient, perplexity, loss -- is handled entirely by pythae's own
`QuantizerEMA`.

As before, since the real strip data (`pileup_ml`) is not available here, we
substitute MNIST (flattened per-image into a single 784-length 1D vector),
following `benchmark_VAE/examples/notebooks/models_training/vqvae_training.ipynb`,
and compute **RMSE only**.

Save/load is provided both the pythae-native way (`model.save` /
`AutoModel.load_from_folder`, used automatically by `TrainingPipeline`) and
as an explicit `save_model` / `load_model` pair mirroring the
`VQVAE.save` / `VQVAE.load` pattern from `vqvae_quantizer.py`
(a `vqvae_metadata.json` config file + a `vqvae_model.pth` state dict).

> Written to be logically complete and consistent, but not executed here --
> run it yourself to validate shapes/hyperparameters.


## 0. Setup

In [ ]:
# If pythae is not installed, install it (or install the local benchmark_VAE checkout instead:
# %pip install -e /path/to/benchmark_VAE)
%pip install pythae


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import torchvision.datasets as datasets
from torch.utils.data import Dataset
from torchvision import transforms
from pileup_ml.strips.hits import DETID_SIZE, StripDigiEvent
from pileup_ml.strips.segments import StripEventSegments, event_hits_to_segments, event_segments_to_hits


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# For reproducible results (mirrors simple_vqvae_ema_strips.ipynb)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True
SEED = 123
torch.manual_seed(SEED)
np.random.seed(SEED)


In [ ]:
from pythae.models.base.base_utils import ModelOutput
from pythae.models.nn import BaseEncoder, BaseDecoder
from pythae.models import VQVAE, VQVAEConfig, AutoModel
from pythae.trainers import BaseTrainerConfig
from pythae.pipelines.training import TrainingPipeline
from pileup_ml.detectors.strips import StripsDetector


In [ ]:
CHANNELS = 1
VECTOR_LENGTH = 8
EVENT_COUNT = 3

## 1. Config

`CustomVQVAEConfig` only adds `hidden_channels` on top of pythae's own
`VQVAEConfig` (which already carries `input_dim`, `latent_dim`,
`num_embeddings`, `use_ema`, `decay`, `commitment_loss_factor`,
`quantization_loss_factor`, ...). `embedding_dim` is left unset here -- pythae
sets `model_config.embedding_dim` itself, from the encoder's actual output
size, inside `VQVAE._set_quantizer`.


In [ ]:
from pydantic.dataclasses import dataclass as pydantic_dataclass


@pydantic_dataclass
class CustomVQVAEConfig(VQVAEConfig):
    """pythae VQVAEConfig + the one extra field needed to build the
    Conv1d encoder/decoder ported from vqvae_layers.py."""
    hidden_channels: int = 16


In [ ]:
detector_info = Path(os.environ['DETID_INFO_DIR'])
strip_detector = StripsDetector.load(detector_info)
strip_events_train = StripDigiEvent.read_root(Path(os.environ['STRIP_ROOT_FILE_DIR']) / '0001_10.root', detector=strip_detector)[:EVENT_COUNT]
strip_events_test = StripDigiEvent.read_root(Path(os.environ['STRIP_ROOT_FILE_DIR']) / '0002_100.root', detector=strip_detector)[:EVENT_COUNT]

## 2. Encoder / Decoder

Ports of `EncoderConv` / `DecoderConv` / `ResBlock` from `vqvae_layers.py`,
using `nn.Conv1d` / `nn.ConvTranspose1d` (the original `ResBlock` /
`EncoderConv` pass a `data_type` kwarg into `torch.nn.Conv2d`, which isn't
part of the public `torch.nn` API and looks like it's meant to be intercepted
by an internal `pileup_ml` shim we don't have here -- the conv sizes,
`GroupNorm`, residual blocks and `AdaptiveAvgPool1d` bottleneck are otherwise
unchanged).

The only architectural change versus `vqvae_layers.py` is at the very
edges of the encoder/decoder: the encoder **flattens** its final
`(latent_dim, ENCODED_PATCH_INDEXES)` feature map into a single 2D vector of
size `latent_dim * ENCODED_PATCH_INDEXES` before returning it, and the
decoder **unflattens** it back -- so that pythae's stock `VQVAE` treats each
whole input vector as being represented by a single codebook entry (rather
than us hand-rolling a spatial/multi-code quantizer), letting pythae's own
`QuantizerEMA` handle quantization end-to-end.


In [ ]:
KERNEL_SIZE = 4
STRIDE = 2
PADDING = 1
ENCODED_PATCH_INDEXES = 4   # matches `encoded_patch_indexes` in vq_spec / ModelConfig


class ResBlock1d(nn.Module):
    """Port of `ResBlock` from vqvae_layers.py, using Conv1d for 1D vector data."""

    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.ReLU(),
            nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv1d(out_channels, in_channels, kernel_size=1, stride=1, padding=0),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.conv_block(x)


class CustomEncoderConv(BaseEncoder):
    """Port of `EncoderConv` from vqvae_layers.py, flattened to a 2D
    embedding so pythae's stock `VQVAE` / `QuantizerEMA` can be used as-is."""

    def __init__(self, model_config: CustomVQVAEConfig) -> None:
        BaseEncoder.__init__(self)
        channels = model_config.input_dim[0]
        hidden_channels = model_config.hidden_channels
        latent_dim = model_config.latent_dim

        self.latent_dim = latent_dim
        self.encoded_patch_indexes = ENCODED_PATCH_INDEXES

        self.model = nn.Sequential(
            nn.Conv1d(
                in_channels=channels,
                out_channels=hidden_channels,
                kernel_size=KERNEL_SIZE,
                stride=STRIDE,
                padding=PADDING,
            ),
            nn.GroupNorm(8, hidden_channels),
            nn.ReLU(inplace=True),
            nn.Conv1d(
                in_channels=hidden_channels,
                out_channels=latent_dim,
                kernel_size=KERNEL_SIZE - 1,
                padding=PADDING,
            ),
        )

        self.adaptive_avg_1d = nn.AdaptiveAvgPool1d(ENCODED_PATCH_INDEXES)

        self.residual = nn.Sequential(
            ResBlock1d(in_channels=latent_dim, out_channels=latent_dim // 2),
            ResBlock1d(in_channels=latent_dim, out_channels=latent_dim // 2),
        )

    def forward(self, x: torch.Tensor) -> ModelOutput:
        out = self.model(x)                     # (N, latent_dim, L)
        out = self.adaptive_avg_1d(out)          # (N, latent_dim, ENCODED_PATCH_INDEXES)
        out = self.residual(out)                 # (N, latent_dim, ENCODED_PATCH_INDEXES)
        out = out.reshape(out.shape[0], -1)       # (N, latent_dim * ENCODED_PATCH_INDEXES)
        return ModelOutput(embedding=out)


class CustomDecoderConv(BaseDecoder):
    """Port of `DecoderConv` from vqvae_layers.py, unflattening the 2D
    quantized vector handed back by pythae's stock `VQVAE.forward`."""

    def __init__(self, model_config: CustomVQVAEConfig) -> None:
        BaseDecoder.__init__(self)
        channels = model_config.input_dim[0]
        hidden_channels = model_config.hidden_channels
        latent_dim = model_config.latent_dim

        self.latent_dim = latent_dim
        self.encoded_patch_indexes = ENCODED_PATCH_INDEXES

        self.adaptive_avg_1d = nn.AdaptiveAvgPool1d(ENCODED_PATCH_INDEXES)

        self.residual = nn.Sequential(
            ResBlock1d(in_channels=latent_dim, out_channels=latent_dim // 2),
            ResBlock1d(in_channels=latent_dim, out_channels=latent_dim // 2),
            nn.ReLU(),
        )

        self.model = nn.Sequential(
            nn.ConvTranspose1d(
                in_channels=latent_dim,
                out_channels=hidden_channels,
                kernel_size=KERNEL_SIZE - 1,
                padding=PADDING,
            ),
            nn.GroupNorm(8, hidden_channels),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(
                in_channels=hidden_channels,
                out_channels=channels,
                kernel_size=KERNEL_SIZE,
                stride=STRIDE,
                padding=PADDING,
            ),
        )

    def forward(self, z: torch.Tensor) -> ModelOutput:
        out = z.reshape(z.shape[0], self.latent_dim, self.encoded_patch_indexes)
        out = self.adaptive_avg_1d(out)   # kept for parity with vqvae_layers.py; no-op here
        out = self.residual(out)
        out = self.model(out)             # (N, channels, segment_size)
        out = torch.sigmoid(out)          # reconstruction in [0, 1], since pythae's
                                           # default VQVAE.loss_function uses plain MSE
                                           # (not BCE-with-logits like vqvae_quantizer.py)
        return ModelOutput(reconstruction=out)


In [ ]:
class AddNormalization:
    
    """
    Helper class for normalizing data from 0 to 1
    """
    
    def __call__(self, x: np.array) -> torch.Tensor:
        x = torch.as_tensor(x)
        x = x / 1023
        x = x.float()
        return x
    
    def __repr__(self) -> str:
        name = self.__class__.__name__
        return f"{name} ()"

In [ ]:
# Strips data prepearation
def hits_to_vectors(self, event: StripDigiEvent) -> StripEventSegments:
        return event_hits_to_segments(
                event,
                segment_size=self.segment_size,
                fill_value=self.fill_value,
                dtype=self.segment_dtype
        ).as_array()

vectors_collected_train = []      

for event_train in strip_events_train:
     vectors = hits_to_vectors(event_train)
     vectors_collected_train.append(vectors)
strip_segments_train = np.concatenate(vectors_collected_train)

vectors_collected_test = []      
for event_test in strip_events_test:
     vectors = hits_to_vectors(event_test)
     vectors_collected_test.append(vectors)
     
strip_segments_test = np.concatenate(vectors_collected_test)

## 3. Data -- MNIST as 1D vectors

Each 28x28 MNIST image is flattened into a single-channel 1D vector of
length 784 (`input_dim = (1, 784)`), scaled to `[0, 1]`, standing in for the
detector-strip vectors used in `simple_vqvae_ema_strips.ipynb`.


In [ ]:
# VECTOR_LENGTH = 28 * 28   # each MNIST image flattened into one 1D vector
# CHANNELS = 1

# mnist_trainset = datasets.MNIST(root="../../data", train=True, download=True, transform=None)

# # mirrors `train_dataset` / `eval_dataset` split in the pythae MNIST example
# train_dataset = mnist_trainset.data[:-10000].reshape(-1, CHANNELS, VECTOR_LENGTH) / 255.0
# eval_dataset = mnist_trainset.data[-10000:].reshape(-1, CHANNELS, VECTOR_LENGTH) / 255.0

# train_dataset = train_dataset.float()
# eval_dataset = eval_dataset.float()

# print(train_dataset.shape, eval_dataset.shape)

class StripSegmentsDataset(Dataset):
    """
    Pytorch dataset class for making PixelEventHit adcs into tensors

    Returns:
        torch.Tensor: adcs patches as tensors
    """
    
    def __init__(self, segments: list[object], transform=None):
        self.segments = segments
        self.transform = transform

    def __len__(self) -> int:
        return len(self.segments)

    def __getitem__(self, index: int) -> np.ndarray | torch.Tensor:
        
        event_segment = torch.as_tensor(self.segments[index])
         
        # Applying the transform
        if self.transform:
            event_segment = self.transform(event_segment)
        
        return event_segment

data_transform = transforms.Compose([
    AddNormalization()
])
vqvae_trainset = StripSegmentsDataset(strip_segments_train, data_transform)
vqvae_testset = StripSegmentsDataset(strip_segments_test, data_transform)

## 4. Model config + instantiation (stock pythae `VQVAE` + `QuantizerEMA`)

In [ ]:
# mirrors `vq_spec` in simple_vqvae_ema_strips.ipynb
model_config = CustomVQVAEConfig(
    input_dim=(CHANNELS, VECTOR_LENGTH),
    latent_dim=32,
    hidden_channels=16,
    num_embeddings=128,
    use_ema=True,
    decay=0.99,
    commitment_loss_factor=0.01,   # plays the role of `beta` in vqvae_quantizer.py
)

encoder = CustomEncoderConv(model_config)
decoder = CustomDecoderConv(model_config)

# Stock pythae VQVAE -- `_set_quantizer` sees a 2D encoder embedding and
# automatically instantiates `QuantizerEMA` (since `use_ema=True`); no
# subclassing / custom quantizer needed.
model = VQVAE(
    model_config=model_config,
    encoder=encoder,
    decoder=decoder,
).to(device)

print(f"embedding_dim resolved by pythae: {model.model_config.embedding_dim}")
print(f"quantizer: {type(model.quantizer).__name__}")
model


## 5. Save / load

pythae already checkpoints the model for you (`TrainingPipeline` writes a
`final_model` folder loadable with `AutoModel.load_from_folder`, used in
section 7 below). In addition, here are explicit `save_model` / `load_model`
helpers mirroring the `VQVAE.save` / `VQVAE.load` pattern from
`vqvae_quantizer.py` (a `vqvae_metadata.json` config file next to a
`vqvae_model.pth` state dict), for parity with that example.


In [ ]:
METADATA_FILENAME = "vqvae_metadata.json"
VQVAE_STATE_FILENAME = "vqvae_model.pth"


def save_model(model: VQVAE, model_dir: Path) -> None:
    """Port of `VQVAE.save` from vqvae_quantizer.py."""
    model_dir = Path(model_dir)
    model_meta_data_path = model_dir / METADATA_FILENAME
    model_state_path = model_dir / VQVAE_STATE_FILENAME

    if not model_dir.exists():
        os.makedirs(model_dir)
        print(f"INFO: {str(model_dir)} folder for saving model states created!")

    torch.save(model.state_dict(), model_state_path)

    config_dict = {
        "type": "vqvae",
        "params": {"config": model.model_config.to_dict()},
    }
    model_meta_data_path.write_text(json.dumps(config_dict))

    print(f"SUCCESS: Model saved in {str(model_state_path)}")


def load_model(model_dir: Path) -> VQVAE:
    """Port of `VQVAE.load` from vqvae_quantizer.py."""
    model_dir = Path(model_dir)
    model_meta_data_path = (model_dir / METADATA_FILENAME).read_text()
    model_state_path = model_dir / VQVAE_STATE_FILENAME

    config_dict = json.loads(model_meta_data_path)
    config = CustomVQVAEConfig(**config_dict["params"]["config"])

    loaded_encoder = CustomEncoderConv(config)
    loaded_decoder = CustomDecoderConv(config)
    loaded_model = VQVAE(model_config=config, encoder=loaded_encoder, decoder=loaded_decoder)

    loaded_model.load_state_dict(torch.load(model_state_path, weights_only=False))
    return loaded_model


## 6. Train with pythae's `TrainingPipeline`

In [ ]:
training_config = BaseTrainerConfig(
    output_dir="my_custom_vqvae_model",
    learning_rate=4e-3,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_epochs=100,   # matches `num_epochs` in vq_spec
)

pipeline = TrainingPipeline(
    training_config=training_config,
    model=model,
)


In [ ]:
pipeline(
    train_data=vqvae_trainset,
    eval_data=vqvae_testset,
)


## 7. Load trained model

Both loading paths are shown: pythae's own `AutoModel.load_from_folder`
(what `TrainingPipeline` produces), and the `load_model` helper from
section 5, which reproduces `VQVAE.load` from `vqvae_quantizer.py` (used
here against a save produced via `save_model`).


In [ ]:
last_training = sorted(os.listdir("vqvae"))[-1]
final_model_dir = os.path.join("vqvae", last_training, "final_model")

trained_model = AutoModel.load_from_folder(final_model_dir).to(device)
trained_model.eval()


In [ ]:
# round-trip demo of the vqvae_quantizer.py-style save/load
save_model(trained_model, Path("vqvae/vqvae_save"))
trained_model = load_model(Path("vqvae/vqvae_save")).to(device)
trained_model.eval()


## 8. RMSE evaluation

Only RMSE is computed here (matching `compare_matching_hits(..., metric='rmse')`
from `simple_vqvae_ema_strips.ipynb`, simplified to a direct RMSE between
original and reconstructed vectors, since the domain-specific hit matching
used for real strip/pixel data does not apply to plain MNIST vectors).


In [ ]:
@torch.no_grad()
def reconstruct(model, data, batch_size=256):
    model.eval()
    outputs = []
    for start in range(0, data.shape[0], batch_size):
        batch = data[start:start + batch_size].to(device)
        out = model({"data": batch})
        outputs.append(out.recon_x.cpu())
    return torch.cat(outputs, dim=0)


reconstructions = reconstruct(trained_model, vqvae_testset)
print(reconstructions.shape)


In [ ]:
def rmse(original: torch.Tensor, reconstructed: torch.Tensor) -> float:
    """Root-mean-square error between original and reconstructed vectors."""
    diff = (original - reconstructed).reshape(original.shape[0], -1)
    per_sample_mse = torch.mean(diff ** 2, dim=1)
    return torch.sqrt(per_sample_mse).mean().item()


overall_rmse = rmse(vqvae_testset, reconstructions)
print(f"RMSE (eval set): {overall_rmse:.6f}")


## 9. Visual sanity check (optional)

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=8, figsize=(16, 4))
for i in range(8):
    axes[0][i].plot(vqvae_testset[i, 0].numpy())
    axes[0][i].set_title("original")
    axes[0][i].axis("off")

    axes[1][i].plot(reconstructions[i, 0].numpy())
    axes[1][i].set_title("reconstruction")
    axes[1][i].axis("off")

plt.tight_layout()
plt.show()
